# Translation of data - NL2EN

In [5]:
%pip install transformers accelerate
%pip install ctranslate2 OpenNMT-py==2.* sentencepiece
%pip install sacremoses

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
#!pip install transformers -U
import transformers

In [8]:
#https://opennmt.net/CTranslate2/guides/transformers.html#marianmt
!ct2-transformers-converter --model Helsinki-NLP/opus-mt-nl-en --output_dir ./../../ct2_model --force

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\p70092940\OneDrive - Maastricht University\Desktop\Projects\MyIBDcoach-NLP\mijnidbcoachnlp\.venv\Scripts\ct2-transformers-converter.exe\__main__.py", line 7, in <module>
  File "c:\Users\p70092940\OneDrive - Maastricht University\Desktop\Projects\MyIBDcoach-NLP\mijnidbcoachnlp\.venv\Lib\site-packages\ctranslate2\converters\transformers.py", line 2482, in main
    converter.convert_from_args(args)
  File "c:\Users\p70092940\OneDrive - Maastricht University\Desktop\Projects\MyIBDcoach-NLP\mijnidbcoachnlp\.venv\Lib\site-packages\ctranslate2\converters\converter.py", line 50, in convert_from_args
    return self.convert(
           ^^^^^^^^^^^^^
  File "c:\Users\p70092940\OneDrive - Maastricht University\Desktop\Projects\MyIBDcoach-NLP\mijnidbcoachnlp\.venv\Lib\site-packages\ctranslate2\converters\converter.py", line 102, in convert
  

In [3]:
# Load dataset
import pandas as pd
import tqdm

path = "..\\..\\dataset\\analysis_datasets\\sentence_data.xlsx"
df = pd.read_excel(path, index_col=0)
messages = df["Sentence"].tolist()

In [4]:
import ctranslate2
import transformers
import tqdm

# Load CTranslate2 Translator and the tokenizer
translator = ctranslate2.Translator('./../../ct2_model')
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-nl-en")

# Function to translate a batch of messages using CTranslate2
def translate_batch(messages, batch_size=10):
    translations = []
    
    for i in tqdm.tqdm(range(0, len(messages), batch_size)):
        batch = messages[i:i + batch_size]
        
        # Tokenize the batch of messages into tokens (lists of token strings)
        source_batch = [tokenizer.convert_ids_to_tokens(tokenizer.encode(message, add_special_tokens=True)) for message in batch]

        # Translate the batch using CTranslate2 (expects a list of tokenized inputs)
        results = translator.translate_batch(source_batch)

        # Decode the hypotheses (translations) back to text
        for result in results:
            translated_tokens = result.hypotheses[0]  # Access the first hypothesis (most likely translation)
            translated_text = tokenizer.convert_tokens_to_string(translated_tokens)  # Convert tokens back to string
            translations.append(translated_text)
        
        # Print original and translated messages
        #for original, translated in zip(batch, translations[-len(batch):]):
            #print(f"Original: {original}")
            #print(f"Translated: {translated}")
    
    return translations

# Function to clean and filter valid text messages
def clean_messages(messages):
    return [str(message) for message in messages if isinstance(message, str) and message.strip()]

# Clean the messages to remove any non-string or empty values
cleaned_messages = clean_messages(messages)

# Translate messages in batches using CTranslate2
translated_messages = translate_batch(cleaned_messages)

# Add the translations back into the DataFrame (optional)
df["Translated_Sentence"] = [None if not isinstance(msg, str) else translated_messages.pop(0) for msg in messages]

# Save the translated messages to a new file (optional)
df.to_excel("..\\..\\dataset\\analysis_datasets\\translated_sentence_data.xlsx", index=False)


RuntimeError: Unable to open file 'model.bin' in model './../../ct2_model'

In [8]:
# read the message df and map the language to the sentences
df = pd.read_excel("..\\..\\dataset\\analysis_datasets\\translated_sentence_data.xlsx", index_col=0)
message_df = pd.read_excel("..\\..\\dataset\\analysis_datasets\\message_data.xlsx", index_col=0)
message_df
df = df.merge(message_df[['Message_ID']], on="Message_ID", how='left')
df

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,0,I was rushed into the [PRESSION-1] [LOCATION-1...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",1,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,2,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",3,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",4,"I got very nauseous, vomiting, dizzy and rejuv..."
...,...,...,...,...,...
42532,33229,"Ze hebben de urine op kweek gezet, kan morgenv...","Ze hebben de urine op kweek gezet, kan morgenv...",42532,"They've put the urine on culture, can get the ..."
42533,33229,Afgelopen jaren is er al vaker bloed in mijn u...,Afgelopen jaren is er al vaker bloed in mijn u...,42533,"In recent years, blood has been found in my ur..."
42534,33229,Wat adviseert u hiermee te doen?,Wat adviseert u hiermee te doen?,42534,What do you suggest you do with this?
42535,33231,"Hoi [PERSOON-1], Oke, dat is iig al een gerust...","Oke, dat is iig al een geruststelling ;) ik wa...",42535,"Hi [PRESSON-1], Okay, that's a relief;) I'll w..."


In [9]:
df.to_excel("..\\..\\dataset\\analysis_datasets\\translated_sentence_data.xlsx", index=False)
